# Aegis Wave - Model Training & Validation
## Optimized for 2-Class Fall Detection on Heterogeneous Edge Devices

This notebook adapts the **Aegis Wave** methodology for real-world CSI data, handling common dataset quirks:
1. **Mismatched subcarriers**: Standardizes both 232-subcarrier (HP) and 64-subcarrier (ESP32) inputs to a 64x500 window.
2. **Data Robustness**: Skips malformed HDF5 files and handles subjects with missing classes.
3. **Split Consistency**: Adheres to the pre-defined ID-based splits (train/val/test).
4. **Edge Ready**: Outputs a quantized TFLite model with high-confidence thresholds.

In [ ]:
import numpy as np
import pandas as pd
import h5py
import os
import json
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from scipy import signal
import matplotlib.pyplot as plt
from datetime import datetime

# Set constants
DATA_DIR = "data/"
SUB_COUNT = 64  # Standardized for both device types
TIME_STEPS = 500
EMERGENCY_THRESHOLD = 0.85
ANOMALY_THRESHOLD = 0.60
LABEL_MAP = {"Fall": 0, "Nonfall": 1}
INV_LABEL_MAP = {0: "Fall", 1: "Nonfall"}

## 1. Data Loading & Robust Processing
We use a generator approach to handle large amounts of raw HDF5 data without exhausting memory.

In [ ]:
def butterworth_filter(data, lowcut=0.5, highcut=10, fs=100, order=4):
    """Apply Butterworth bandpass filter to isolate human movement"""
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype="band")
    return signal.filtfilt(b, a, data, axis=1)

def load_csi_sample(file_path):
    """Robust HDF5 loading with shape standardization"""
    full_path = os.path.join(DATA_DIR, file_path.lstrip("./"))
    try:
        with h5py.File(full_path, "r") as f:
            data = f["CSI_amps"][:]
            data = data.squeeze() 
            if data.shape[0] >= SUB_COUNT:
                data = data[:SUB_COUNT, :]
            else:
                return None
            if data.shape[1] < TIME_STEPS:
                return None
            data = data[:, :TIME_STEPS]
            data = butterworth_filter(data)
            return data.T 
    except Exception:
        return None

def load_dataset_from_ids(ids_file, metadata_df):
    """Loads all valid samples belonging to a split ID set"""
    with open(os.path.join(DATA_DIR, "splits", ids_file), "r") as f:
        ids = json.load(f)
    X, y = [], []
    for sample_id in ids:
        row = metadata_df[metadata_df["id"] == sample_id]
        if row.empty: continue
        sample = load_csi_sample(row.iloc[0]["file_path"])
        if sample is not None:
            X.append(sample)
            y.append(LABEL_MAP[row.iloc[0]["label"]])
    return np.array(X), np.array(y)

metadata_df = pd.read_csv(os.path.join(DATA_DIR, "metadata/sample_metadata.csv"))
X_train, y_train = load_dataset_from_ids("train_id.json", metadata_df)
X_val, y_val = load_dataset_from_ids("val_id.json", metadata_df)
X_test, y_test = load_dataset_from_ids("test_id.json", metadata_df)
print(f"Loaded Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val.reshape(X_val.shape[0], -1)).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test.reshape(X_test.shape[0], -1)).reshape(X_test.shape)
y_train_cat = keras.utils.to_categorical(y_train, 2)
y_val_cat = keras.utils.to_categorical(y_val, 2)
y_test_cat = keras.utils.to_categorical(y_test, 2)


In [ ]:
model = keras.Sequential([
    keras.layers.Conv1D(32, 5, activation="relu", input_shape=(TIME_STEPS, SUB_COUNT)),
    keras.layers.MaxPooling1D(2),
    keras.layers.Conv1D(64, 3, activation="relu"),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(2, activation="softmax")
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
model.fit(X_train_scaled, y_train_cat, validation_data=(X_val_scaled, y_val_cat), epochs=20, batch_size=32)


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open("aegis_wave_final.tflite", "wb") as f: f.write(tflite_model)
print("✅ TFLite model saved.")